In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import re

In [2]:
df = pd.read_csv('../data/preprocessed_data/news_data.csv')
print(df.shape)
df["date"]=pd.to_datetime(df["date"]).dt.normalize()
df.head()


(4658, 11)


,company,ticker,text,created,nid,article_title,article_url,info_index,datetime,datetime_my,date
0,Hartalega,5168.KL,Hartalega Holdings Bhd ’s (KL: HARTA ) net pro...,1794496907000,825567,"Pharmaniaga, Hartalega, Johor Plantations, Key...",https://theedgemalaysia.com/node/825567,2,2026-11-12 15:21:47+00:00,2026-11-12 23:21:47+08:00,2026-11-12
1,Dialog Group,7277.KL,Dialog Group Bhd (KL: DIALOG ) said it is inve...,1794407094000,825408,"Zetrix AI, Dialog, United Plantations, Heineke...",https://theedgemalaysia.com/node/825408,2,2026-11-11 14:24:54+00:00,2026-11-11 22:24:54+08:00,2026-11-11
2,Sunway,5211.KL,Sunway Real Estate Investment Trust (KL: SUNRE...,1794323122000,825220,"Zetrix AI, Sunway REIT, Hup Seng, YNH Property...",https://theedgemalaysia.com/node/825220,3,2026-11-10 15:05:22+00:00,2026-11-10 23:05:22+08:00,2026-11-10
3,Public Bank,1295.KL,Public Bank Bhd (KL: PBBANK ) saw its net prof...,1793975243000,824927,"Unisem, Frontken, Public Bank, Maxis, Zetrix A...",https://theedgemalaysia.com/node/824927,3,2026-11-06 14:27:23+00:00,2026-11-06 22:27:23+08:00,2026-11-06
4,Maxis,6012.KL,Maxis Bhd (KL: MAXIS ) on Friday reported a 8....,1793975243000,824927,"Unisem, Frontken, Public Bank, Maxis, Zetrix A...",https://theedgemalaysia.com/node/824927,4,2026-11-06 14:27:23+00:00,2026-11-06 22:27:23+08:00,2026-11-06


In [3]:
display(df.info())
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4658 entries, 0 to 4657
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   company        4658 non-null   object        
 1   ticker         4658 non-null   object        
 2   text           4658 non-null   object        
 3   created        4658 non-null   int64         
 4   nid            4658 non-null   int64         
 5   article_title  4658 non-null   object        
 6   article_url    4658 non-null   object        
 7   info_index     4658 non-null   int64         
 8   datetime       4658 non-null   object        
 9   datetime_my    4658 non-null   object        
 10  date           4658 non-null   datetime64[ns]
dtypes: datetime64[ns](1), int64(3), object(7)
memory usage: 400.4+ KB


None

,created,nid,info_index,date
count,4.658000e+03,4658.000000,4658.000000,4658
mean,1.595973e+12,511202.647918,9.961786,2020-07-28 15:38:52.503220224
min,1.414764e+12,167488.000000,0.000000,2014-10-31 00:00:00
25%,1.488214e+12,335227.000000,2.000000,2017-02-28 00:00:00
50%,1.596728e+12,525339.500000,7.000000,2020-08-06 00:00:00
75%,1.692569e+12,679472.750000,13.000000,2023-08-20 12:00:00
max,1.794497e+12,825567.000000,97.000000,2026-11-12 00:00:00
std,1.132968e+11,193859.625312,10.623985,NaN


In [4]:
from transformers import pipeline
finbert = pipeline("sentiment-analysis",model="ProsusAI/finbert")

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

c:\Users\bilal\Desktop\ProjetPython\FinAgent-X An Explainable Multi-Agent Financial Intelligence Platform\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bilal\.cache\huggingface\hub\models--ProsusAI--finbert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [5]:
def analyze_sentiment(text):

    text = str(text)
    text = " ".join(text.split())

    results = finbert(
        text,
        top_k=None,
        truncation=True
    )

    scores = {
        item["label"]: item["score"]
        for item in results
    }

    sentiment = max(scores,key=scores.get)

    sentiment_score = (
        scores["positive"]
        - scores["negative"]
    )

    return {
        "sentiment": sentiment,
        "positive_score": scores["positive"],
        "negative_score": scores["negative"],
        "neutral_score": scores["neutral"],
        "sentiment_score": sentiment_score
    }

In [6]:
from tqdm import tqdm
for index, row in tqdm(df.iterrows()):
    text = row["text"]
    sentiment_results = analyze_sentiment(text)
    df.at[index, "sentiment"] = sentiment_results["sentiment"]
    df.at[index, "positive_score"] = sentiment_results["positive_score"]
    df.at[index, "negative_score"] = sentiment_results["negative_score"]
    df.at[index, "neutral_score"] = sentiment_results["neutral_score"]
    df.at[index, "sentiment_score"] = sentiment_results["sentiment_score"]
    

4658it [09:03,  8.57it/s]


In [7]:
df[
    [
        "company",
        "sentiment",
        "positive_score",
        "negative_score",
        "neutral_score",
        "sentiment_score"
    ]
].head()

,company,sentiment,positive_score,negative_score,neutral_score,sentiment_score
0,Hartalega,positive,0.953960,0.021876,0.024165,0.932084
1,Dialog Group,neutral,0.189534,0.007492,0.802974,0.182043
2,Sunway,negative,0.012582,0.973576,0.013842,-0.960994
3,Public Bank,negative,0.009825,0.976235,0.013940,-0.966410
4,Maxis,positive,0.955980,0.018319,0.025701,0.937661


In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

df["n_tokens"] = df["text"].apply(
    lambda x: len(tokenizer(str(x),add_special_tokens=True)["input_ids"])
)

df["n_tokens"].describe()

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1236 > 512). Running this sequence through the model will result in indexing errors


count    4658.000000
mean       98.487978
std        66.112988
min        12.000000
25%        56.000000
50%        82.000000
75%       129.000000
max      1735.000000
Name: n_tokens, dtype: float64

In [9]:
(df["n_tokens"] > 512).sum()

np.int64(5)

In [10]:
df["sentiment"].value_counts(normalize=True) * 100

sentiment
positive    48.905109
negative    26.685273
neutral     24.409618
Name: proportion, dtype: float64

In [11]:
pd.crosstab(
    df["company"],
    df["sentiment"]
)

sentiment,negative,neutral,positive
company,,,
99 Speed Mart,2,2,7
AMMB,29,30,37
Alliance Bank,9,6,18
Axiata,51,56,72
Bursa Malaysia,25,11,25
CIMB,50,57,96
CelcomDigi,29,22,32
Dialog Group,25,22,53
Farm Fresh,5,7,9


In [12]:
df.groupby("company")["sentiment_score"].mean().sort_values(
    ascending=False
)

company
Frontken                      0.608194
Sunway Construction           0.491853
99 Speed Mart                 0.445998
Hong Leong Bank               0.399928
TIME dotCom                   0.384199
Sunway                        0.380130
Gamuda                        0.351575
ViTrox                        0.347713
MR DIY                        0.339345
Nestle Malaysia               0.338524
Press Metal                   0.338523
Maxis                         0.333675
IJM                           0.322975
Greatech                      0.321833
Dialog Group                  0.300228
Sime Darby Property           0.283630
KPJ Healthcare                0.269634
Tenaga Nasional               0.257643
Alliance Bank                 0.257525
Hartalega                     0.252280
Maybank                       0.243065
IHH Healthcare                0.232068
QL Resources                  0.220879
MISC                          0.218666
Public Bank                   0.216561
Westports        

In [ ]:
df.to_csv("../data/preprocessed_data/news_data_sentiment.csv",index=False)